# Coachly - FunctionGemma Fine-tune (Colab T4)
Notebook minimale: genera dataset -> allena QLoRA -> valida output JSON.

In [ ]:
# Imposta il path del progetto in Colab
REPO_DIR = '/content/voice-ml-recognizer'
import os
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Manca {REPO_DIR}. Carica il repo o rinomina il path.')
%cd {REPO_DIR}

In [ ]:
!nvidia-smi
!python -V
!pip install -q -r refactor/requirements-colab.txt

In [ ]:
# Genera dataset nuovo (qualita > quantita)
!python refactor/dataset_creator.py --output_dir refactor/data --per_action_per_lang 340 --unknown_per_lang 170
!python - <<'PY'
import json
from pathlib import Path
m = json.loads(Path('refactor/data/metadata.json').read_text(encoding='utf-8'))
print('Sizes:', m['sizes'])
print('Actions all:', m['stats']['all']['action'])
PY

In [ ]:
# Se conosci il model id preciso di FunctionGemma, inseriscilo qui
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = 'refactor/output/functiongemma_qlora'
cmd = f"python refactor/colab_functiongemma_train.py --data_dir refactor/data --output_dir {OUTPUT_DIR}"
if BASE_MODEL:
    cmd += f" --base_model {BASE_MODEL}"
print(cmd)
!$cmd

In [ ]:
# Leggi i risultati rapidi
import json
from pathlib import Path
q = Path('refactor/output/functiongemma_qlora/quick_eval.json')
if q.exists():
    print(json.dumps(json.loads(q.read_text(encoding='utf-8')), indent=2))
else:
    print('quick_eval.json non trovato')

In [ ]:
# Eval su frasi STT sporche (typo, filler, code-switch)
import json
import os
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER_DIR = 'output/functiongemma_qlora/adapter'
META_PATH = 'data/metadata.json'

if not os.path.isdir(ADAPTER_DIR):
    raise FileNotFoundError(f'Adapter non trovato: {ADAPTER_DIR}. Prima completa il training.')

if os.path.exists(META_PATH):
    system_prompt = json.loads(open(META_PATH, encoding='utf-8').read())['system_prompt']
else:
    system_prompt = (
        'You are Coachly NLU. Convert workout speech-to-text into strict JSON. '
        'Return ONLY valid JSON, no markdown.'
    )

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading tokenizer/model...')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map='auto',
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()


def extract_json(s: str):
    s = s.strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    m = re.search(r'\{.*\}', s, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def predict(text: str):
    msgs = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': text},
    ]
    inp = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids=inp,
            max_new_tokens=180,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_ids = out[0][inp.shape[-1]:]
    txt = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    parsed = extract_json(txt)
    return txt, parsed


noisy_cases = [
    ('aggiungii bencc press 3x10 a cedimnto', 'ADD_EXERCISE'),
    ('fatto deadlif 5 rep 140 kg', 'LOG_SET'),
    ('rimouvi lat mascin', 'DELETE_EXERCISE'),
    ('no aspetta aggiungi panca piana 3x8 e trazioni 4x6', 'ADD_EXERCISE'),
    ('uhm metti squat 4 serie da 8 e push ap 3x15 drop set', 'ADD_EXERCISE'),
    ('correggo togli pushup e rematore', 'DELETE_EXERCISE'),
    ('modifca deadlift a 4x6 120kg', 'UPDATE_SET'),
    ('aggiorna trazioni 5x5 con pausa', 'UPDATE_SET'),
    ('ho fatto bench press 8 rep 80 kilo to failur', 'LOG_SET'),
    ('done benh press 3 set of 8 rep 80 kg', 'LOG_SET'),
    ('add squat 5x5 100kg and pull up 4x6', 'ADD_EXERCISE'),
    ('wait no actually remove lath pull down', 'DELETE_EXERCISE'),
    ('change deadlift to 3x5 at 140 kg', 'UPDATE_SET'),
    ('how many calories i burn today', 'UNKNOWN'),
    ('fammi una scheda petto tricipiti pls', 'UNKNOWN'),
    ('metti timer 90 secondi', 'UNKNOWN'),
    ('aggiungi leg pres 4x10 e stacco rumno 3x8', 'ADD_EXERCISE'),
    ('done squat 1x5 120 kg amrap', 'LOG_SET'),
    ('delete bench and incline benh', 'DELETE_EXERCISE'),
    ('update push up to 4x20', 'UPDATE_SET'),
]

ok = 0
valid = 0
for i, (text, expected) in enumerate(noisy_cases, start=1):
    raw, parsed = predict(text)
    pred_action = parsed.get('action') if isinstance(parsed, dict) else None
    is_valid = isinstance(parsed, dict)
    is_ok = pred_action == expected
    valid += int(is_valid)
    ok += int(is_ok)
    print(f'[{i:02d}] expected={expected:16s} pred={str(pred_action):16s} valid_json={is_valid} ok={is_ok}')
    print(f'  text: {text}')
    if is_valid:
        print(f'  json: {json.dumps(parsed, ensure_ascii=False)}')
    else:
        print(f'  raw: {raw}')
    print()

print('---')
print(f'Action accuracy on noisy set: {ok}/{len(noisy_cases)} = {ok/len(noisy_cases):.1%}')
print(f'Valid JSON rate: {valid}/{len(noisy_cases)} = {valid/len(noisy_cases):.1%}')

